# CoordGen playground — two molecules

Edit **Inputs**, re-run **Render**. Logic is in `notebooks/_coordgen_align.py`
(tested by `tests/test_coordgen_align_notebook.py`).

| Field | Meaning |
| --- | --- |
| `REF_SMILES` / `QUERY_SMILES` | Molecules |
| `MATCH` | `(query_idx, ref_idx)` pairs, or `None` for auto MCS / substructure |
| `CONSTRAINT` | `"constrained"` / `"fixed"` / `"none"` |
| `FLAGS` | Native CoordGen knobs |

The **OVERLAY** panel shares one viewBox (gray=ref, black=query). Auto-fit
single-molecule SVGs cannot prove alignment.

```bash
make build-python
uv run pytest -q tests/test_coordgen_align_notebook.py
uv run jupyter notebook notebooks/coordgen_playground.ipynb
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import HTML, display
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

_NB = Path.cwd()
if (_NB / "_coordgen_align.py").is_file():
    sys.path.insert(0, str(_NB))
elif (_NB / "notebooks" / "_coordgen_align.py").is_file():
    sys.path.insert(0, str(_NB / "notebooks"))
else:
    raise RuntimeError("run from repo root or notebooks/")

from _coordgen_align import (
    layout_xy,
    matched_rms,
    mol_no_h,
    native_aligned,
    overlay_svg,
    paint_svg,
)
from xpict import _native
from xpict.layout2d import (
    LAYOUT_FLAG_BEST_PRECISION,
    LAYOUT_FLAG_EVEN_ANGLES,
    LAYOUT_FLAG_FORCE_OPEN_MACROCYCLES,
    LAYOUT_FLAG_METAL_BONDS_AS_ZOB,
    LAYOUT_FLAG_QUICK_PRECISION,
    LAYOUT_FLAG_SKIP_MINIMIZATION,
)

assert _native.HAS_COORDGEN, "make build-python"

FLAG_BITS = {
    "even_angles": LAYOUT_FLAG_EVEN_ANGLES,
    "quick_precision": LAYOUT_FLAG_QUICK_PRECISION,
    "best_precision": LAYOUT_FLAG_BEST_PRECISION,
    "force_open_macrocycles": LAYOUT_FLAG_FORCE_OPEN_MACROCYCLES,
    "metal_bonds_as_zob": LAYOUT_FLAG_METAL_BONDS_AS_ZOB,
    "skip_minimization": LAYOUT_FLAG_SKIP_MINIMIZATION,
}


def layout_flags_from(flags: dict[str, bool]) -> int:
    bits = 0
    for name, on in flags.items():
        if on:
            bits |= FLAG_BITS[name]
    return bits


def numbered_svg(mol: Chem.Mol, highlight: set[int] | None = None, w: int = 280, h: int = 220) -> str:
    m = Chem.Mol(mol)
    for a in m.GetAtoms():
        a.SetProp("atomNote", str(a.GetIdx()))
    Chem.rdDepictor.Compute2DCoords(m)
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    rdMolDraw2D.PrepareAndDrawMolecule(d, m, highlightAtoms=list(highlight or []))
    d.FinishDrawing()
    return d.GetDrawingText()


def show_row(title: str, panels: list[tuple[str, str]]):
    parts = [
        f"<h3>{title}</h3>",
        "<div style='display:flex;gap:1rem;flex-wrap:wrap;align-items:flex-start'>",
    ]
    for label, svg in panels:
        parts.append(
            f"<div><div style='font:12px sans-serif;margin-bottom:4px'>{label}</div>{svg}</div>"
        )
    parts.append("</div>")
    display(HTML("".join(parts)))

print("ready")

## Inputs

In [ ]:
REF_SMILES = "c1ccccc1O"
QUERY_SMILES = "COc1ccccc1"
MATCH: list[tuple[int, int]] | None = None
CONSTRAINT = "fixed"  # "constrained" | "fixed" | "none"
FLAGS = dict(
    even_angles=False,
    quick_precision=False,
    best_precision=False,
    force_open_macrocycles=False,
    metal_bonds_as_zob=False,
    skip_minimization=False,
)

## Render

In [ ]:
flags = layout_flags_from(FLAGS)
fixed = CONSTRAINT == "fixed"
use_map = CONSTRAINT != "none"

ref_m, query_m, ref_free, query_free, query_aligned, pairs = native_aligned(
    REF_SMILES,
    QUERY_SMILES,
    fixed=fixed,
    layout_flags=flags,
    pairs=MATCH if use_map else [],
)
if not use_map:
    query_aligned = query_free
    pairs = []
elif MATCH is None:
    pass  # pairs from native_aligned

rms = matched_rms(query_aligned, ref_free, pairs) if pairs else float("nan")
print(f"ref={ref_m.GetNumAtoms()} query={query_m.GetNumAtoms()} mapped={len(pairs)}")
print(f"MATCH={pairs}")
print(f"CONSTRAINT={CONSTRAINT!r} flags={flags:#x} RMS={rms:.4f}")
if CONSTRAINT == "fixed" and pairs:
    assert rms < 1e-2, f"hard RMS {rms}"

show_row(
    "Atom indices",
    [
        ("ref", numbered_svg(ref_m, {r for _, r in pairs})),
        ("query", numbered_svg(query_m, {q for q, _ in pairs})),
    ],
)
show_row(
    "Auto-fit SVGs (not an alignment proof)",
    [
        ("ref free", paint_svg(ref_m, ref_free)),
        ("query free", paint_svg(query_m, query_free)),
        (f"query {CONSTRAINT}", paint_svg(query_m, query_aligned)),
    ],
)
show_row(
    "OVERLAY shared viewBox (gray=ref, black=query)",
    [
        (
            f"RMS={rms:.4f}",
            overlay_svg(
                ref_m,
                layout_xy(ref_free),
                query_m,
                layout_xy(query_aligned),
                pairs,
                title=f"{CONSTRAINT}",
            ),
        )
    ],
)

## Presets — run one, then re-run Render

In [ ]:
REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccccc1O", "COc1ccccc1", None, "fixed"
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccccc1", "Cc1ccccc1", None, "fixed"
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "c1ccc2ccccc2c1", "c1ccc2ncccc2c1", None, "fixed"
# E/Z
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = r"C/C=C\C", r"C/C=C\CC", None, "fixed"
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = r"C/C=C/C", r"C/C=C/CCC", None, "fixed"
# Chiral
# REF_SMILES, QUERY_SMILES, MATCH, CONSTRAINT = "C[C@H](O)C(=O)O", "C[C@H](N)C(=O)O", None, "fixed"
# FLAGS["even_angles"] = True
print(REF_SMILES, "→", QUERY_SMILES, CONSTRAINT)
